In [ ]:
# Ekstrak Landmark utuk Angka
import cv2
import mediapipe as mp
import os
import csv
import numpy as np
import random

DATASET_PATH = "dataset_angka"
OUTPUT = "angka3.csv"
TARGET_PER_CLASS = 400  # lebih realistis

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.6  # 🔥 lebih stabil
)

# ==========================================================
# AUGMENTASI (LEBIH AMAN)
# ==========================================================
def augment_image(img):
    h, w = img.shape[:2]
    aug = img.copy()

    # 🔹 brightness & contrast ringan
    alpha = random.uniform(0.9, 1.1)
    beta  = random.randint(-15, 15)
    aug = cv2.convertScaleAbs(aug, alpha=alpha, beta=beta)

    # 🔹 rotasi kecil
    angle = random.uniform(-10, 10)
    M = cv2.getRotationMatrix2D((w // 2, h // 2), angle, 1.0)
    aug = cv2.warpAffine(aug, M, (w, h), borderMode=cv2.BORDER_REFLECT)

    # 🔹 zoom ringan
    scale = random.uniform(0.95, 1.0)
    cx, cy = w // 2, h // 2
    nw, nh = int(w * scale), int(h * scale)
    x1 = max(cx - nw // 2, 0)
    y1 = max(cy - nh // 2, 0)
    x2 = min(x1 + nw, w)
    y2 = min(y1 + nh, h)
    aug = cv2.resize(aug[y1:y2, x1:x2], (w, h))

    return aug

# ==========================================================
# EKSTRAK LANDMARK (VERSI LEBIH KUAT)
# ==========================================================
def extract_landmarks(rgb_img):
    result = hands.process(rgb_img)

    if not result.multi_hand_landmarks:
        return None

    landmarks = result.multi_hand_landmarks[0]

    # 🔥 pakai RELATIVE POSITION (lebih stabil)
    base_x = landmarks.landmark[0].x
    base_y = landmarks.landmark[0].y

    data = []

    for lm in landmarks.landmark:
        data.append(lm.x - base_x)
        data.append(lm.y - base_y)

    # 🔥 NORMALISASI
    data = np.array(data)
    data = data - np.min(data)
    if np.max(data) != 0:
        data = data / np.max(data)

    # 🔥 FILTER DATA JELEK
    if np.max(data) - np.min(data) < 0.05:
        return None

    # 🔥 TAMBAHAN FITUR JARAK JARI
    lm = landmarks.landmark
    def dist(a, b):
        return ((a.x - b.x)**2 + (a.y - b.y)**2) ** 0.5

    data = data.tolist()
    data.append(dist(lm[4], lm[8]))
    data.append(dist(lm[8], lm[12]))
    data.append(dist(lm[12], lm[16]))
    data.append(dist(lm[16], lm[20]))

    return data

# ==========================================================
# MAIN
# ==========================================================
with open(OUTPUT, 'w', newline='') as f:
    writer = csv.writer(f)

    for label in sorted(os.listdir(DATASET_PATH)):
        folder = os.path.join(DATASET_PATH, label)
        if not os.path.isdir(folder):
            continue

        img_files = [
            os.path.join(folder, fn)
            for fn in os.listdir(folder)
            if fn.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))
        ]

        if not img_files:
            print(f"[{label}] tidak ada gambar")
            continue

        print(f"[{label}] mulai dari {len(img_files)} gambar")

        # 🎯 balance ringan
        if str(label) == "6":
            TARGET = 200
        else:
            TARGET = TARGET_PER_CLASS

        berhasil = 0
        gagal = 0
        attempts = 0
        MAX_ATTEMPTS = TARGET * 10

        while berhasil < TARGET and attempts < MAX_ATTEMPTS:
            attempts += 1

            src_path = random.choice(img_files)
            img = cv2.imread(src_path)
            if img is None:
                continue

            img = cv2.resize(img, (640, 480))

            # 🔥 augmentasi lebih seimbang
            if random.random() < 0.5:
                processed = img.copy()
            else:
                processed = augment_image(img)

            rgb = cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)
            data = extract_landmarks(rgb)

            if data:
                data.append(label)
                writer.writerow(data)
                berhasil += 1

                if berhasil % 50 == 0:
                    print(f"   {berhasil}/{TARGET}")
            else:
                gagal += 1

        print(f"[{label}] ✅ {berhasil}/{TARGET} | gagal: {gagal} | attempts: {attempts}\n")

hands.close()
print("✅ selesai:", OUTPUT)